# Limpieza y Modelado Logistico con Random Forest

Notebook para cargar `dataset_logistico.csv`, limpiar las variables necesarias y entrenar un modelo `RandomForestRegressor` para predecir `sale_amount`.

## 1. Librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 2. Carga del dataset

In [ ]:
RUTA_DATASET = "../data/dataset_logistico.csv"

df = pd.read_csv(RUTA_DATASET)

print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")
display(df.head())

## 3. Revision inicial

In [ ]:
df.info()

print("\nValores nulos por columna:")
display(df.isna().sum().sort_values(ascending=False).to_frame("nulos"))

print(f"\nFilas duplicadas: {df.duplicated().sum():,}")

display(df.describe(include="all").T)

## 4. Limpieza y preparacion

In [ ]:
df = df.copy()

df["dt"] = pd.to_datetime(df["dt"], errors="coerce")

# Eliminamos filas con problemas basicos en fecha o variable objetivo.
df = df.dropna(subset=["dt", "sale_amount"]).copy()

# Eliminamos duplicados exactos.
df = df.drop_duplicates().copy()

# Codificacion simple con pandas para variables categoricas.
columnas_categoricas = [
    "product_id",
    "store_id",
    "first_category_id",
    "second_category_id",
    "third_category_id",
    "holiday_flag",
    "activity_flag",
]

for col in columnas_categoricas:
    if col in df.columns:
        df[col] = df[col].astype("category")
        df[f"{col}_cod"] = df[col].cat.codes

print(f"Filas despues de limpieza: {len(df):,}")
display(df.head())

## 5. Variables para el modelo

In [ ]:
TARGET = "sale_amount"

FEATURES = [
    "product_id_cod",
    "store_id_cod",
    "first_category_id_cod",
    "second_category_id_cod",
    "third_category_id_cod",
    "discount",
    "holiday_flag_cod",
    "activity_flag_cod",
    "mes",
    "dia_mes",
    "dia_semana",
    "fin_semana",
    "horas_con_stock",
    "venta_lag_1",
    "venta_lag_7",
    "venta_promedio_7d",
    "venta_promedio_14d",
    "stock_lag_1",
]

features_existentes = [col for col in FEATURES if col in df.columns]

df_modelo = df[["dt", TARGET] + features_existentes].dropna().sort_values("dt").reset_index(drop=True)

print("Target:", TARGET)
print("Features utilizadas:")
for feature in features_existentes:
    print("-", feature)

print(f"\nFilas finales para modelado: {len(df_modelo):,}")
display(df_modelo.head())

## 6. Division cronologica train-test

In [ ]:
punto_corte = int(len(df_modelo) * 0.80)

train = df_modelo.iloc[:punto_corte].copy()
test = df_modelo.iloc[punto_corte:].copy()

X_train = train[features_existentes]
y_train = train[TARGET]
X_test = test[features_existentes]
y_test = test[TARGET]

print(f"Train: {len(train):,} filas")
print(f"Test: {len(test):,} filas")
print(f"Fecha minima train: {train['dt'].min()}")
print(f"Fecha maxima train: {train['dt'].max()}")
print(f"Fecha minima test: {test['dt'].min()}")
print(f"Fecha maxima test: {test['dt'].max()}")

## 7. Entrenamiento con RandomForestRegressor

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Modelo entrenado correctamente")

## 8. Evaluacion

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

resultados = pd.DataFrame({
    "dt": test["dt"].values,
    "real": y_test.values,
    "pred": y_pred,
    "error_abs": np.abs(y_test.values - y_pred)
})

display(resultados.head())

## 9. Graficos del modelo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.3, color="#185FA5")
limites = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(limites, limites, "r--")
axes[0].set_title("Real vs Predicho")
axes[0].set_xlabel("Valor real")
axes[0].set_ylabel("Valor predicho")

residuos = y_test.values - y_pred
sns.histplot(residuos, bins=40, kde=True, ax=axes[1], color="#378ADD")
axes[1].set_title("Distribucion de residuos")
axes[1].set_xlabel("Residuo")

plt.tight_layout()
plt.show()

## 10. Importancia de variables

In [ ]:
importancias = pd.Series(rf.feature_importances_, index=features_existentes).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importancias.values[:12], y=importancias.index[:12], color="#185FA5")
plt.title("Top variables mas importantes")
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

display(importancias.to_frame("importancia"))